# Official EXO-200 v1 Transformer Training

Runs the standardized 3-tokenization × 2-position-encoding classification matrix. EXOBench remains authoritative for loading, preprocessing, run-level splitting, optimization, checkpoint selection, and test evaluation.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
import torch

configured_root = os.environ.get('EXO200_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'exo200_detector',
    Path.cwd(),
    Path.cwd().parent / 'exo200_detector',
    Path.cwd().parent,
    Path.cwd().parent.parent / 'exo200_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'exo_transformer').is_dir()
     and (candidate / 'exobench').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate exo200_detector; set EXO200_TRANSFORMER_PROJECT_ROOT.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from exobench import (
    DataConfig, TrainingConfig, evaluate_model, prepare_dataset,
    set_seed, train_model,
)
from exo_transformer import (
    EXOTransformerClassifier, TokenizationConfig, validate_split_manifest,
)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('EXOBench:', Path(sys.modules['exobench'].__file__).resolve())
print('Transformer:', Path(sys.modules['exo_transformer'].__file__).resolve())

In [ ]:
DATA_ROOT = Path(os.environ.get(
    'EXO200_BENCH_DATA',
    str(PROJECT_ROOT.parent / 'data' / 'EXO-200'),
)).expanduser()
OUTPUT_ROOT = Path(os.environ.get(
    'EXO200_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
SUMMARY_PATH = OUTPUT_ROOT / 'transformer_results.csv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_config = DataConfig(
    data_root=DATA_ROOT,
    validation_fraction=0.10,
    baseline_samples=200,
    classification_amplitude_normalization=True,
    seed=42,
)
training_config = TrainingConfig()  # Shared collaborator defaults.

# Add a run ID only when intentionally treating it as already complete.
COMPLETED_OFFICIAL_RUNS = set()

print('Dataset:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Data config:', data_config.to_dict())
print('Training config:', training_config.to_dict())

In [ ]:
EXPERIMENTS = [
    {'tokenization': tokenization, 'position_encoding': position_encoding}
    for tokenization in ('raw_patches', 'segment_summary', 'pulse_entities')
    for position_encoding in ('coordinate_mlp', 'fourier_coordinates')
]
ALL_RUN_IDS = {
    f'classification__{row["tokenization"]}__{row["position_encoding"]}'
    for row in EXPERIMENTS
}
requested = os.environ.get('EXO200_RUN_IDS', '').strip()
SELECTED_RUN_IDS = (
    {value.strip() for value in requested.split(',') if value.strip()}
    if requested else set()
)
unknown = SELECTED_RUN_IDS - ALL_RUN_IDS
if unknown:
    raise ValueError(f'Unknown EXO200_RUN_IDS: {sorted(unknown)}')

def make_tokenization_config(name):
    return TokenizationConfig(
        tokenization=name,
        channel_regions=7,
        time_regions=12,
        uniform_entity_fraction=0.5,
        entity_context_size=9,
    )

def run_is_complete(run_dir):
    required = (
        'run_config.json', 'best.pt', 'history.json',
        'metrics.json', 'predictions.npz', 'run_summary.json',
    )
    return all((run_dir / name).is_file() for name in required)

def reset_training_shuffle(data, seed):
    # EXOBench wraps its DataLoader in a prefetch adapter. Resetting the
    # loader-owned generator gives every representation the same epoch-1
    # shuffle sequence while keeping the dataset prepared only once.
    loader = getattr(data.train_loader, 'loader', data.train_loader)
    generator = getattr(loader, 'generator', None)
    if generator is None:
        raise RuntimeError('EXOBench training loader has no seeded generator')
    generator.manual_seed(seed)

def upsert_summary(row):
    if SUMMARY_PATH.is_file():
        table = pd.read_csv(SUMMARY_PATH)
        table = table.loc[table['run_id'] != row['run_id']]
    else:
        table = pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    table = table.sort_values(['tokenization', 'position_encoding'])
    table.to_csv(SUMMARY_PATH, index=False)

experiment_table = pd.DataFrame([
    {
        'run_id': f'classification__{row["tokenization"]}__{row["position_encoding"]}',
        **row,
    }
    for row in EXPERIMENTS
])
display(experiment_table)
print('Selected runs:', sorted(SELECTED_RUN_IDS) if SELECTED_RUN_IDS else 'all')

In [ ]:
print('Preparing the official EXO-200 v1 run-level split once...')
classification_data = prepare_dataset(
    data_config=data_config,
    batch_size=training_config.batch_size,
    num_workers=training_config.num_workers,
)
classification_data.require_two_classes()

split_manifest = validate_split_manifest(
    classification_data, data_root=DATA_ROOT
)
print('Split manifest:', split_manifest['dataset_doi'])

print('Counts:', classification_data.counts)
print('Class counts:', classification_data.class_counts)
print('Runs:', classification_data.runs)
print('Overlap:', classification_data.overlap_counts)

In [ ]:
for experiment in EXPERIMENTS:
    tokenization = experiment['tokenization']
    position_encoding = experiment['position_encoding']
    run_id = f'classification__{tokenization}__{position_encoding}'
    run_dir = OUTPUT_ROOT / run_id

    if SELECTED_RUN_IDS and run_id not in SELECTED_RUN_IDS:
        continue
    if run_id in COMPLETED_OFFICIAL_RUNS or run_is_complete(run_dir):
        print(f'Skipping completed run: {run_id}')
        continue

    print('\n' + '=' * 88)
    print(run_id)
    print('=' * 88)
    run_dir.mkdir(parents=True, exist_ok=True)
    tokenization_config = make_tokenization_config(tokenization)

    set_seed(training_config.seed, training_config.deterministic)
    reset_training_shuffle(classification_data, training_config.seed)
    model = EXOTransformerClassifier(
        tokenization_config=tokenization_config,
        position_encoding=position_encoding,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        num_frequencies=6,
    )
    parameter_count = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    )
    run_config = {
        'run_id': run_id,
        'dataset': 'EXO-200 v1',
        'task': 'classification',
        'data': data_config.to_dict(),
        'counts': classification_data.counts,
        'class_counts': classification_data.class_counts,
        'split_runs': classification_data.runs,
        'split_overlap_counts': classification_data.overlap_counts,
        'training': training_config.to_dict(),
        'representation': model.config_dict(),
        'parameter_count': parameter_count,
    }
    (run_dir / 'run_config.json').write_text(
        json.dumps(run_config, indent=2), encoding='utf-8'
    )
    print('Trainable parameters:', f'{parameter_count:,}')

    training_start = time.perf_counter()
    history = train_model(
        model,
        classification_data.train_loader,
        classification_data.validation_loader,
        config=training_config,
        output_dir=run_dir,
    )
    training_seconds = time.perf_counter() - training_start

    evaluation_start = time.perf_counter()
    metrics = evaluate_model(
        model,
        classification_data.test_loader,
        device=training_config.device,
        output_dir=run_dir,
        use_amp=training_config.use_amp,
        amp_precision=training_config.amp_precision,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start
    checkpoint = torch.load(
        run_dir / 'best.pt', map_location='cpu', weights_only=False
    )
    epochs_completed = len(history)
    row = {
        'run_id': run_id,
        'task': 'classification',
        'tokenization': tokenization,
        'position_encoding': position_encoding,
        'parameter_count': parameter_count,
        'train_events': classification_data.counts['train'],
        'validation_events': classification_data.counts['validation'],
        'test_events': classification_data.counts['test'],
        'epochs_completed': epochs_completed,
        'best_epoch': int(checkpoint['epoch']),
        'best_validation_auc': float(checkpoint['score']),
        'training_seconds': training_seconds,
        'minutes_per_epoch': training_seconds / max(epochs_completed, 1) / 60.0,
        'evaluation_seconds': evaluation_seconds,
        'test_auc': metrics.get('auc'),
        'test_accuracy': metrics.get('accuracy'),
        'test_loss': metrics.get('loss'),
    }
    (run_dir / 'run_summary.json').write_text(
        json.dumps(row, indent=2, allow_nan=True), encoding='utf-8'
    )
    upsert_summary(row)
    print(json.dumps(row, indent=2, allow_nan=True))

In [ ]:
results = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.is_file() else pd.DataFrame()
print('Completed official runs:', len(results), '/ 6')
display(results)